In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import pandas as pd
import torch
import torch.nn as nn

from src.dataset import make_loaders
from src.evaluate import predict_loader, summarize
from src.model import CNNBaseline
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()

## Controlled ablations: 

In [ ]:
def save_result(row):
    new = pd.DataFrame([row])
    if os.path.exists(RESULTS):
        old = pd.read_csv(RESULTS)
        new = pd.concat([old[old['experiment'] != row['experiment']], new])
    new.to_csv(RESULTS, index=False)

def run_experiment(name, augment=False, dropout_p=0.0, pool_type='max',
                   weight_decay=0.0, scheduler_name='none', lr=1e-3, 
                   seed=SEED, epochs=20): 
    set_seed(seed)
    train_ds, val_ds, train_loader, val_loader = make_loaders(
        DATA, IMG_SIZE, BATCH, seed, augment)
    model = CNNBaseline(dropout_p=dropout_p, pool_type=pool_type).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler, step_on_val = make_scheduler(scheduler_name, optimizer)
    criterion = nn.CrossEntropyLoss()

    config = {
        'augment': augment, 'dropout_p': dropout_p, 'pool_type': pool_type,
        'weight_decay': weight_decay, 'scheduler': scheduler_name, 'lr': lr,
        'epochs': epochs,   
        'batch_size': BATCH, 'optimizer': 'AdamW'
    }
    meta = {
        'architecture': 'CNNBaseline',
        'class_to_idx': train_ds.class_to_idx,
        'img_size': IMG_SIZE, 'seed': seed, 'config': config
    }
    ckpt = f'../checkpoints/{name}.pt'
    history = fit(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs,  
        ckpt, f'../logs/{name}.csv',
        meta=meta, scheduler=scheduler, step_on_val=step_on_val
    )
    model.load_state_dict(torch.load(ckpt, weights_only=False)['model_state_dict'])
    trues, preds = predict_loader(model, val_loader, device)
    row = summarize(trues, preds, train_ds.classes)
    best = max(history, key=lambda r: r['val_macro_f1'])
    row.update({
        'experiment': name, 'best_epoch': best['epoch'],
        'train_loss': best['train_loss'], 'val_loss': best['val_loss'],
        'gap': best['val_loss'] - best['train_loss'],
        'final_lr': history[-1]['lr'], **config
    })
    save_result(row)
    return row

In [ ]:
run_experiment('baseline')

In [ ]:
run_experiment('aug' , augment= True)

In [8]:
run_experiment('dropout_0.3', dropout_p=0.3)

{'epoch': 1, 'train_loss': 1.9743021417547155, 'val_loss': 1.8653093344205385, 'val_macro_f1': 0.1964603552580944, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.8250389334596233, 'val_loss': 1.7953900074029898, 'val_macro_f1': 0.24459929342210282, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.753970001950676, 'val_loss': 1.7452569032644296, 'val_macro_f1': 0.2800449947332446, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.7345278098104526, 'val_loss': 1.6634018204428933, 'val_macro_f1': 0.32196993519463324, 'lr': 0.001}
{'epoch': 5, 'train_loss': 1.6433229784906647, 'val_loss': 1.6131651082596221, 'val_macro_f1': 0.35683242570704926, 'lr': 0.001}
{'epoch': 6, 'train_loss': 1.6466375129264699, 'val_loss': 1.591737784967794, 'val_macro_f1': 0.41817848785985245, 'lr': 0.001}
{'epoch': 7, 'train_loss': 1.5699811799716763, 'val_loss': 1.5594200474875315, 'val_macro_f1': 0.3505159282189656, 'lr': 0.001}
{'epoch': 8, 'train_loss': 1.5241530728541541, 'val_loss': 1.5397618572433274, 'val_macro_f1': 0.40512

{'macro_precision': np.float64(0.5793522591510738),
 'macro_recall': np.float64(0.5640712651920858),
 'macro_f1': np.float64(0.5587126081208018),
 'lowest_recall_class': 'kamyunet',
 'lowest_precision_class': 'autobus',
 'recall_ambulance': np.float64(0.6136363636363636),
 'precision_ambulance': np.float64(0.5625),
 'recall_autobus': np.float64(0.5306122448979592),
 'precision_autobus': np.float64(0.40625),
 'recall_kamyun': np.float64(0.56),
 'precision_kamyun': np.float64(0.6511627906976745),
 'recall_kamyunet': np.float64(0.2127659574468085),
 'precision_kamyunet': np.float64(0.45454545454545453),
 'recall_minibus': np.float64(0.5555555555555556),
 'precision_minibus': np.float64(0.4807692307692308),
 'recall_savari': np.float64(0.66),
 'precision_savari': np.float64(0.4647887323943662),
 'recall_taxi': np.float64(0.86),
 'precision_taxi': np.float64(0.8269230769230769),
 'recall_vanet': np.float64(0.52),
 'precision_vanet': np.float64(0.7878787878787878),
 'experiment': 'dropout_0.

In [ ]:
run_experiment('dropout_0.5', dropout_p=0.5)

{'epoch': 1, 'train_loss': 1.9944697056907272, 'val_loss': 1.8856207364565367, 'val_macro_f1': 0.19180763344195728, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.8531045150725853, 'val_loss': 1.8133946573579467, 'val_macro_f1': 0.22825874443368838, 'lr': 0.001}
